# Downloading dataset and connecting to drive

In [ ]:
# ==============================================================================
# PART 0 & 1: INSTALL DEPENDENCIES, SETUP DRIVE AND PROJECT DIRECTORY
# ==============================================================================
!pip install -q ultralytics deep-sort-realtime roboflow

import os
from google.colab import drive
from ultralytics import YOLO
from roboflow import Roboflow

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Dog_Tracker_Enhanced'
DATASETS_DIR = os.path.join(PROJECT_DIR, 'datasets')
TRAINING_DIR = os.path.join(PROJECT_DIR, 'training_runs')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs')

os.makedirs(DATASETS_DIR, exist_ok=True)
os.makedirs(TRAINING_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd {DATASETS_DIR}

# ==============================================================================
# PART 2: DOWNLOAD DATASET & TRAIN ENHANCED YOLOv8 MODEL
# ==============================================================================
# NOTE: Replace with your own Roboflow API key if you have one.
rf = Roboflow(api_key="vuWWYVpCntUbWnPc1Xd8")
project = rf.workspace("nftlab").project("stray-dogs-4h56u-vmx98")
dataset = project.version(1).download("yolov8")

DATASET_YAML_PATH = os.path.join(dataset.location, 'data.yaml')
print(f"Dataset downloaded to: {dataset.location}")


ERROR: Operation cancelled by user


KeyboardInterrupt: 

# Training the model

In [ ]:
# TIER 2 IMPROVEMENT: Use the more accurate 'yolov8s.pt' (small) model as a base
model = YOLO('yolov8s.pt')

print("\nStarting model training (50 epochs with yolov8s)...")
results = model.train(
    data=DATASET_YAML_PATH,
    epochs=50, # TIER 2 IMPROVEMENT: Train for more epochs for better accuracy
    imgsz=640,
    project=TRAINING_DIR,
    name='dog_detector_yolov8s',
    exist_ok=True
)
print("Training complete.")


Starting model training (50 epochs with yolov8s)...
Ultralytics 8.3.227 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Dog_Tracker_Enhanced/datasets/Stray-Dogs-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=dog_detector_yolov8s, nbs=64, nms=False

# **Using Trained Model on our input videos**

In [2]:
!pip install -q ultralytics deep-sort-realtime roboflow

import os
from google.colab import drive
from ultralytics import YOLO
from roboflow import Roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 136.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
# ==============================================================================
# PART 3: THE TRACKING AND BEHAVIOR ANALYSIS PIPELINE (FINAL CORRECTED VERSION 6)
# ==============================================================================

# --- Imports and Initialization ---
import os
import cv2
import json
import numpy as np
import itertools
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from google.colab.patches import cv2_imshow
import math

print("Initializing models and paths...")
PROJECT_DIR = '/content/drive/MyDrive/Dog_Tracker_Enhanced'
TRAINING_DIR = os.path.join(PROJECT_DIR, 'training_runs')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs')
CUSTOM_MODEL_PATH = os.path.join(TRAINING_DIR, 'dog_detector_yolov8s/weights/best.pt')

if not os.path.exists(CUSTOM_MODEL_PATH):
    raise FileNotFoundError(f"FATAL: Trained model not found at {CUSTOM_MODEL_PATH}.")

model = YOLO(CUSTOM_MODEL_PATH)
tracker = DeepSort(max_age=50, n_init=5, nms_max_overlap=1.0)
print("Successfully loaded custom model and initialized tracker with tuned parameters.")

# --- Class IDs and Configuration ---
DOG_CLASS_IDS = [0, 2, 3, 4, 5, 6]
PERSON_CLASS_ID = 7
CONF_THRESHOLD = 0.40

# --- Video and Log Paths ---
VIDEO_SOURCE_PATH = '/content/drive/MyDrive/3521475477-preview.mp4'
OUTPUT_VIDEO_PATH = os.path.join(OUTPUT_DIR, 'output_video_final_stable_2.mp4')
LOG_JSONL_PATH = os.path.join(OUTPUT_DIR, 'tracking_log_final_stable.jsonl')

# --- Open video files and log ---
cap = cv2.VideoCapture(VIDEO_SOURCE_PATH)
if not cap.isOpened(): raise IOError(f"Cannot open video file: {VIDEO_SOURCE_PATH}")
frame_width, frame_height, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
out_vid = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
log_f = open(LOG_JSONL_PATH, 'w')

# --- Data structures and helper functions ---
track_history = {}

def euclidean_distance(p1, p2): return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def classify_behavior(track_history_data, target_center, W=10, v_stationary=2.0, v_toward=3.0, g_thresh=1.02):
    if len(track_history_data) < 2: return 'unknown', 0.0
    hist = track_history_data[-W:]; speeds, v_rads, a_ratios = [], [], []; target_cx, target_cy = target_center
    for k in range(1, len(hist)):
        p0, p1 = hist[k-1], hist[k]; dx, dy = p1['cx'] - p0['cx'], p1['cy'] - p0['cy']; speeds.append(np.sqrt(dx*dx + dy*dy))
        dir_to_target = np.array([target_cx - p1['cx'], target_cy - p1['cy']]); norm = np.linalg.norm(dir_to_target)
        v_rads.append((dx * dir_to_target[0] + dy * dir_to_target[1]) / norm if norm > 1e-6 else 0.0)
        a_ratios.append((p1['w']*p1['h']) / (p0['w']*p0['h'] + 1e-6))

    # Check if lists are empty before calculating mean
    mean_speed = np.mean(speeds) if speeds else 0.0
    mean_vrad = np.mean(v_rads) if v_rads else 0.0
    mean_area_ratio = np.mean(a_ratios) if a_ratios else 1.0

    if mean_speed < v_stationary and abs(mean_area_ratio - 1.0) < 0.02: return 'stationary', 0.8
    elif mean_vrad > v_toward and mean_area_ratio > g_thresh: return 'moving_toward_user', 0.9
    else: return 'moving_elsewhere', 0.7


def predict_trajectory(hist, n_frames=15, smooth_window=5):
    if len(hist) < smooth_window: return []
    pts = hist[-smooth_window:]; vx_sum, vy_sum = 0, 0
    for i in range(1, len(pts)): vx_sum += pts[i]['cx'] - pts[i-1]['cx']; vy_sum += pts[i]['cy'] - pts[i-1]['cy']
    vx = vx_sum / (len(pts) - 1); vy = vy_sum / (len(pts) - 1)
    traj = []; cx, cy = pts[-1]['cx'], pts[-1]['cy']
    for _ in range(n_frames): cx += vx; cy += vy; traj.append((int(cx), int(cy)))
    return traj

def get_bbox_dist(b1, b2): dx = max(0,b1[0]-b2[2],b2[0]-b1[2]); dy = max(0,b1[1]-b2[3],b2[1]-b1[3]); return math.sqrt(dx*dx+dy*dy)

# --- Main processing loop ---
frame_idx = 0
print("\nStarting video processing with final corrected logic...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    dog_count, stationary_count, interacting_count = 0, 0, 0
    results = model(frame, verbose=False)[0]
    detections_for_deepsort = []
    for box in results.boxes:
        conf, cls = box.conf[0].item(), int(box.cls[0].item())
        internal_cls = 0 if cls in DOG_CLASS_IDS else 1 if cls == PERSON_CLASS_ID else -1
        if internal_cls != -1 and conf > CONF_THRESHOLD:
            x1, y1, x2, y2 = map(int, box.xyxy[0]); w, h = x2 - x1, y2 - y1
            detections_for_deepsort.append(([x1, y1, w, h], conf, internal_cls))

    tracks = tracker.update_tracks(detections_for_deepsort, frame=frame)
    vis_frame = frame.copy()

    user_centroid = (frame_width / 2, frame_height / 2)
    for t in tracks:
        if t.is_confirmed() and t.det_class == 1: user_centroid = ((t.to_tlbr()[0]+t.to_tlbr()[2])/2, (t.to_tlbr()[1]+t.to_tlbr()[3])/2); break

    frame_log = []; frame_visuals = {}
    for track in tracks:
        if not track.is_confirmed() or track.time_since_update > 0: continue

        tid, cid, bbox = track.track_id, track.det_class, track.to_tlbr()
        x1, y1, x2, y2 = map(int, bbox); conf = track.get_det_conf() or 0.0

        if tid not in track_history: track_history[tid] = []
        track_history[tid].append({'frame': frame_idx, 'cx': (x1+x2)/2, 'cy': (y1+y2)/2, 'w': x2-x1, 'h': y2-y1})
        if len(track_history[tid]) > 60: track_history[tid].pop(0)

        label, predicted_traj, trail_points = "tracking...", [], []
        if cid == 0:
            dog_count += 1; color = (0, 255, 0)
            # ### FINAL FIX: Removed the unexpected 'fps' argument from the function call ###
            label, conf = classify_behavior(track_history[tid], user_centroid)
            predicted_traj = predict_trajectory(track_history[tid])
            trail_points = np.array([ (p['cx'], p['cy']) for p in track_history[tid][-20:] ], dtype=np.int32)
        else: color = (255, 0, 0); label = "person"

        frame_visuals[tid] = {'bbox': bbox, 'color': color, 'label': label, 'conf': conf, 'traj': predicted_traj, 'trail': trail_points, 'class_id': cid}

    dog_tracks = {tid: data for tid, data in frame_visuals.items() if data['class_id'] == 0}
    for (id1, d1), (id2, d2) in itertools.combinations(dog_tracks.items(), 2):
        if get_bbox_dist(d1['bbox'], d2['bbox']) < 50:
            frame_visuals[id1]['label'], frame_visuals[id2]['label'] = 'interacting', 0.95
            interacting_count = len({id1, id2})

    for tid, data in frame_visuals.items():
        x1, y1, x2, y2 = map(int, data['bbox'])
        cv2.rectangle(vis_frame, (x1, y1), (x2, y2), data['color'], 2)
        cv2.putText(vis_frame, f"ID:{tid} {data['label']} ({data['conf']:.2f})", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, data['color'], 2)
        if len(data['traj']) > 1: cv2.polylines(vis_frame, [np.array(data['traj'], dtype=np.int32)], False, (0,0,255), 2)
        if len(data['trail']) > 1: cv2.polylines(vis_frame, [data['trail']], False, (255,255,0), 2)
        if data['label'] == 'stationary': stationary_count += 1

        original_class_id = [k for k, v in model.names.items() if v == ('dog' if data['class_id'] == 0 else 'person')][0]
        frame_log.append({'frame': frame_idx, 'id': tid, 'class': model.names[original_class_id], 'bbox': [x1,y1,x2,y2], 'label': data['label'], 'conf': data['conf'], 'traj': data['traj']})

    overlay = vis_frame.copy(); cv2.rectangle(overlay, (5, 5), (250, 100), (0,0,0), -1)
    vis_frame = cv2.addWeighted(overlay, 0.6, vis_frame, 0.4, 0)
    cv2.putText(vis_frame, f"Dogs Tracked: {dog_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(vis_frame, f"Stationary: {stationary_count}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(vis_frame, f"Interacting: {interacting_count}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    out_vid.write(vis_frame)
    if frame_idx % 50 == 0: print(f"Processed frame {frame_idx}, tracking {len(tracks)} objects.")
    for item in frame_log: log_f.write(json.dumps(item) + '\n')
    frame_idx += 1

# --- Cleanup ---
cap.release()
out_vid.release()
log_f.close()
print(f"\nProcessing finished. Output video saved to: {OUTPUT_VIDEO_PATH}")

Initializing models and paths...
Successfully loaded custom model and initialized tracker with tuned parameters.

Starting video processing with final corrected logic...
Processed frame 0, tracking 0 objects.
Processed frame 50, tracking 1 objects.
Processed frame 100, tracking 4 objects.
Processed frame 150, tracking 3 objects.
Processed frame 200, tracking 4 objects.
Processed frame 250, tracking 3 objects.
Processed frame 300, tracking 4 objects.
Processed frame 350, tracking 2 objects.
Processed frame 400, tracking 2 objects.

Processing finished. Output video saved to: /content/drive/MyDrive/Dog_Tracker_Enhanced/outputs/output_video_final_stable_2.mp4
